## 3.1 Build the Baseline Dataset

- Load the cleaned V1 dataset.
- Convert code diffs into TF-IDF feature vectors.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import pandas as pd
train = pd.read_csv('../../data/processed/train_v1.csv')
test = pd.read_csv('../../data/processed/test.csv')
X_train_raw = train['diff']
X_test_raw = test['diff']
y_train = train['top_level_label']
y_test = test['top_level_label']


vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Counter for y_train: {Counter(y_train)}')
print(f'Counter for y_test: {Counter(y_test)}')


X_train shape: (28512, 5793)
X_test shape: (7129, 5793)
Counter for y_train: Counter({'call': 13731, 'expression': 5932, 'control_flow': 4315, 'assignment': 3669, 'identifier': 865})
Counter for y_test: Counter({'call': 3434, 'expression': 1483, 'control_flow': 1079, 'assignment': 917, 'identifier': 216})


## 3.2 Training and Testing the Logistic Regression Model

- Train the Logistic Regression classifier.
- Make predictions on the test set.
- Compare the predictions to the ground truth labels.
- Evaluate the model using the selected metrics.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

lr_model = LogisticRegression(max_iter= 1000)
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

accuracy = accuracy_score(y_test, lr_predictions)
print(f"Accuracy: {accuracy:.4f}")

cf_mat = confusion_matrix(y_test, lr_predictions, labels=lr_model.classes_)
print("Classes:", lr_model.classes_)
print("Confusion Matrix:")
print(cf_mat)

report = classification_report(y_test, lr_predictions)
print(f'Report: {report}')

Accuracy: 0.7081
Classes: ['assignment' 'call' 'control_flow' 'expression' 'identifier']
Confusion Matrix:
[[ 689  150   18   58    2]
 [ 126 3059   29  207   13]
 [  30   49  931   66    3]
 [ 340  738   75  318   12]
 [  66   81    6   12   51]]
Report:               precision    recall  f1-score   support

  assignment       0.55      0.75      0.64       917
        call       0.75      0.89      0.81      3434
control_flow       0.88      0.86      0.87      1079
  expression       0.48      0.21      0.30      1483
  identifier       0.63      0.24      0.34       216

    accuracy                           0.71      7129
   macro avg       0.66      0.59      0.59      7129
weighted avg       0.68      0.71      0.68      7129



## 3.3 Training and Testing the Linear SVM Model

- Repeat the same training and evaluation process as Section 2.2.
- Train the Linear SVM classifier.
- Make predictions on the test set.
- Compare the predictions to the ground truth labels.
- Evaluate the model using the selected metrics.

In [3]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(max_iter= 1000)
svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, svm_predictions)
print(f"Accuracy: {accuracy:.4f}")

cf_mat = confusion_matrix(y_test, svm_predictions, labels=svm_model.classes_)
print("Classes:", svm_model.classes_)
print("Confusion Matrix:")
print(cf_mat)

report = classification_report(y_test, svm_predictions)
print(f'Report: {report}')

Accuracy: 0.6997
Classes: ['assignment' 'call' 'control_flow' 'expression' 'identifier']
Confusion Matrix:
[[ 599  196   21   95    6]
 [  91 3078   44  203   18]
 [  31   60  944   39    5]
 [ 275  791   90  314   13]
 [  53   90    8   12   53]]
Report:               precision    recall  f1-score   support

  assignment       0.57      0.65      0.61       917
        call       0.73      0.90      0.80      3434
control_flow       0.85      0.87      0.86      1079
  expression       0.47      0.21      0.29      1483
  identifier       0.56      0.25      0.34       216

    accuracy                           0.70      7129
   macro avg       0.64      0.58      0.58      7129
weighted avg       0.67      0.70      0.67      7129



## 3.4 Saving the Model and the Vectorizer

- Save both the trained model and the fitted TF-IDF vectorizer.
- When making predictions on new code diffs, the same vectorizer must be used to transform the input before passing it to the model.
- Saving these objects improves reusability, allows other users to run inference without retraining the model, and prepares the project for deployment in a future API.

In [4]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(vectorizer, "../../models/tfidf_vectorizer_v1.joblib")

#Logistic Regression Model
joblib.dump(lr_model, "../../models/logistic_regression_v1.joblib")

#Linear SVC Model
joblib.dump(svm_model, "../../models/linear_svc_v1.joblib")

['../../models/linear_svc_v1.joblib']